In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier,plot_tree 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,log_loss,balanced_accuracy_score
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
import os
os.chdir("/home/pgcp-ai/MachineLearning/Cases/Kyphosis/")

In [2]:
kyp = pd.read_csv("Kyphosis.csv")
kyp
X, y = kyp.drop('Kyphosis', axis = 1), kyp['Kyphosis']

In [3]:
kyp

,Kyphosis,Age,Number,Start
0,absent,71,3,5
1,absent,158,3,14
2,present,128,4,5
3,absent,2,5,1
4,absent,1,4,15
...,...,...,...,...
76,present,157,3,13
77,absent,26,7,13
78,absent,120,2,13
79,present,42,7,6


In [4]:
kyp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Kyphosis  81 non-null     object
 1   Age       81 non-null     int64 
 2   Number    81 non-null     int64 
 3   Start     81 non-null     int64 
dtypes: int64(3), object(1)
memory usage: 2.7+ KB


In [5]:
kyp.isna().sum()

Kyphosis    0
Age         0
Number      0
Start       0
dtype: int64

In [6]:
X,y = kyp.drop("Kyphosis", axis = 1), kyp["Kyphosis"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3, random_state = 26, stratify = y)

In [37]:
lr = LogisticRegression()
knn = KNeighborsClassifier(n_neighbors=3)
lda = LinearDiscriminantAnalysis()
nb = GaussianNB()
dtc = DecisionTreeClassifier(max_depth=4,random_state=26)

voting = VotingClassifier(estimators = [
     (
        "TREE",
        dtc
    ),
     (
        "KNN",
        knn
    ),
    (
        "LR",
        lr
    ),
   
    (
        "LDA",
        lda
    ),
    (
        "NB",
        nb
    ),
   
], voting = 'soft', weights = [1,0.7,7,5,4])
voting.fit(X_train,y_train)

VotingClassifier(estimators=[('TREE',
                              DecisionTreeClassifier(max_depth=4,
                                                     random_state=26)),
                             ('KNN', KNeighborsClassifier(n_neighbors=3)),
                             ('LR', LogisticRegression()),
                             ('LDA', LinearDiscriminantAnalysis()),
                             ('NB', GaussianNB())],
                 voting='soft', weights=[1, 0.7, 7, 5, 4])

In [38]:
estimators = [dtc, knn, lr, lda, nb]
for e in estimators:
    e.fit(X_train, y_train)
    y_pred_prob = e.predict_proba(X_test)
    print("Log Loss of", e, "=", log_loss(y_test, y_pred_prob))

Log Loss of DecisionTreeClassifier(max_depth=4, random_state=26) = 4.582748472683514
Log Loss of KNeighborsClassifier(n_neighbors=3) = 6.0123482471692045
Log Loss of LogisticRegression() = 0.3357707715068842
Log Loss of LinearDiscriminantAnalysis() = 0.3428105961059756
Log Loss of GaussianNB() = 0.39698492604441077


In [39]:
y_pred = voting.predict(X_test)
accuracy_score(y_test,y_pred)

0.84

In [40]:
balanced_accuracy_score(y_test,y_pred)

0.675

### To use `predict_proba` function with `VotingClassifier` we must set `voting='soft'` otherwise it will throw an error

In [41]:
y_pred_prob = voting.predict_proba(X_test)

In [42]:
log_loss(y_test, y_pred_prob)

0.3615679356171432